<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#News-Data" data-toc-modified-id="News-Data-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>News Data</a></span><ul class="toc-item"><li><span><a href="#Import-News-Data-(15-mnths)" data-toc-modified-id="Import-News-Data-(15-mnths)-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Import News Data (15 mnths)</a></span></li><li><span><a href="#Preprocessing" data-toc-modified-id="Preprocessing-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Preprocessing</a></span><ul class="toc-item"><li><span><a href="#Filter-into-ESG-News" data-toc-modified-id="Filter-into-ESG-News-1.2.1"><span class="toc-item-num">1.2.1&nbsp;&nbsp;</span>Filter into ESG News</a></span></li><li><span><a href="#Fix-the-timestamp" data-toc-modified-id="Fix-the-timestamp-1.2.2"><span class="toc-item-num">1.2.2&nbsp;&nbsp;</span>Fix the timestamp</a></span></li></ul></li><li><span><a href="#Convert-into-vector-representation-(Fin-Bert)" data-toc-modified-id="Convert-into-vector-representation-(Fin-Bert)-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Convert into vector representation (Fin-Bert)</a></span><ul class="toc-item"><li><span><a href="#Use-ProsusAI/finbert" data-toc-modified-id="Use-ProsusAI/finbert-1.3.1"><span class="toc-item-num">1.3.1&nbsp;&nbsp;</span>Use ProsusAI/finbert</a></span></li><li><span><a href="#Use-yiyanghkust/finbert-pretrain" data-toc-modified-id="Use-yiyanghkust/finbert-pretrain-1.3.2"><span class="toc-item-num">1.3.2&nbsp;&nbsp;</span>Use yiyanghkust/finbert-pretrain</a></span></li></ul></li></ul></li><li><span><a href="#ESG-Data" data-toc-modified-id="ESG-Data-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>ESG Data</a></span><ul class="toc-item"><li><span><a href="#Match-News-with-ESG-Controversy-Score" data-toc-modified-id="Match-News-with-ESG-Controversy-Score-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Match News with ESG Controversy Score</a></span></li><li><span><a href="#Training-and-Test-Set" data-toc-modified-id="Training-and-Test-Set-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Training and Test Set</a></span></li><li><span><a href="#Data-Preparation's-Done" data-toc-modified-id="Data-Preparation's-Done-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Data Preparation's Done</a></span></li></ul></li><li><span><a href="#Regression-Model" data-toc-modified-id="Regression-Model-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Regression Model</a></span><ul class="toc-item"><li><span><a href="#Choose-three-Models-(Research-First)" data-toc-modified-id="Choose-three-Models-(Research-First)-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Choose three Models (Research First)</a></span></li><li><span><a href="#Cross-Validation" data-toc-modified-id="Cross-Validation-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>Cross Validation</a></span></li><li><span><a href="#GridSearch" data-toc-modified-id="GridSearch-3.3"><span class="toc-item-num">3.3&nbsp;&nbsp;</span>GridSearch</a></span></li><li><span><a href="#Best-Model-with-tuned-Hyperparams" data-toc-modified-id="Best-Model-with-tuned-Hyperparams-3.4"><span class="toc-item-num">3.4&nbsp;&nbsp;</span>Best Model with tuned-Hyperparams</a></span></li></ul></li><li><span><a href="#Numeric-to-Grade-(Refinitiv'-Rule-Based)" data-toc-modified-id="Numeric-to-Grade-(Refinitiv'-Rule-Based)-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Numeric to Grade (Refinitiv' Rule Based)</a></span><ul class="toc-item"><li><span><a href="#Predict-the-Test-Set" data-toc-modified-id="Predict-the-Test-Set-4.1"><span class="toc-item-num">4.1&nbsp;&nbsp;</span>Predict the Test Set</a></span></li></ul></li><li><span><a href="#Archive" data-toc-modified-id="Archive-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Archive</a></span><ul class="toc-item"><li><span><a href="#Classification-FinBert" data-toc-modified-id="Classification-FinBert-5.1"><span class="toc-item-num">5.1&nbsp;&nbsp;</span>Classification FinBert</a></span></li><li><span><a href="#Filterout-ESG-News" data-toc-modified-id="Filterout-ESG-News-5.2"><span class="toc-item-num">5.2&nbsp;&nbsp;</span>Filterout ESG News</a></span></li><li><span><a href="#FinBert-ESG" data-toc-modified-id="FinBert-ESG-5.3"><span class="toc-item-num">5.3&nbsp;&nbsp;</span>FinBert-ESG</a></span></li></ul></li></ul></div>

In [ ]:
import torch
from transformers import BertTokenizer, BertModel

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import svm
from sklearn.naive_bayes import GaussianNB, BernoulliNB, ComplementNB
from sklearn.model_selection import GridSearchCV
from datetime import datetime, timedelta

import pandas as pd
import string
import os
import warnings

pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore')

# download FTSE100
FTSE100_df = pd.read_csv('../data/FTSE100.csv')

# News Data

## Import News Data (15 mnths)

In [ ]:
news_df = pd.read_csv('../data/output_news_headline_FTSE100.csv')
news_df.head()

In [ ]:
news_df[['firm','sourceCode']].groupby('firm').count().sort_values('sourceCode', ascending=False).T

## Preprocessing

### Filter into ESG News

In [ ]:
# dummy dataframe of headline for sentiment result
esg_glossaries = pd.read_csv('../data/ESG Score Compile - Glossary.csv')                 

# remove unused row number
esg_glossaries = esg_glossaries.drop(['No'], axis=1)

# lower the glossaries
esg_glossaries['Keywords'] = esg_glossaries['Keywords'].str.lower()

# remove duplicates
esg_glossaries = esg_glossaries.drop_duplicates(['Keywords'])

In [ ]:
def ESG_news_filter(company_df, text_col='text', inplace=True):
    datapath=os.getcwd()[:-len('notebook')] + "data/"
    ESG_data=pd.read_csv(datapath+'Categorized_ESG_data.csv')
    
    X=ESG_data['T'].str.lower()
    y=ESG_data['S']
    
    cv = CountVectorizer()
    features = cv.fit_transform(X)
    
    pre_trained_model = GaussianNB()
    pre_trained_model.fit(features.toarray(),y)
    
    X_company=company_df[text_col]
    Xf_company=cv.transform(X_company)
    
    ESG=pre_trained_model.predict(Xf_company.toarray())
    
    company_df['ESG']=ESG
    
    if inplace==True:
        return company_df[company_df['ESG']==1].drop(columns='ESG')
    else:
        return company_df


In [ ]:
def filter_esg_news(df, esg_glossaries, column, verbose = False):

    ESG_news_filter = pd.DataFrame()
    cnt = 0
    for key in esg_glossaries.Keywords:
        cnt+=1
        ESG_news_filter = pd.concat([ESG_news_filter, df[df[column].str.contains(key)]])

        if verbose:
            if cnt % 10 == 0:
                print(cnt)
            
    return ESG_news_filter

In [ ]:
def filter_companies(df):
    copy_df = df.copy()
    
    firm_names = [f.split(' ')[0].lower() for f in FTSE100_df['Name']]
    copy_df['flag'] = 0
    
    for firm in firm_names:
        
        index = copy_df[copy_df['text_pp'].str.contains(firm)].index.tolist()
        copy_df.loc[index,'flag'] = 1
        
    return copy_df

In [ ]:
def text_preprocessing_v2(df, types = 1):
    df_copy = df.copy()
    print('total news:',df_copy.shape[0])

    # assign a new seperate column text_pp for preprocessing
    df_copy['text_pp'] = df_copy.text

    # make the string lower
    df_copy['text_pp'] = df_copy['text_pp'].apply(lambda x: x.lower())

    # remove the punctuation
    # translator = str.maketrans(string.punctuation, ' '*len(string.punctuation)) #map punctuation to space
    # df_copy['text_pp'] = df_copy['text_pp'].apply(lambda x: x.translate(translator))

    # remove unnecessary word
    # df_copy['text_pp'] = df_copy['text_pp'].apply(lambda x: x.replace('rpt',''))

    # delete unnecessary double-space
    df_copy['text_pp'] = df_copy['text_pp'].apply(lambda x: ' '.join(x.split()))
    
    if types == 1:
        
        # filter out only ESG news (model)
        df_copy = ESG_news_filter(df_copy, text_col='text_pp', inplace=True)
        print('filter out non-esg news:', df_copy.shape[0])
        
    else:
        
        # filter out only ESG news (conventional)
        df_copy = filter_esg_news(df_copy, esg_glossaries, 'text_pp')
        print('filter out non-esg news:', df_copy.shape[0])
        
    # remove duplicates of headers
    df_copy = df_copy.drop_duplicates(['text_pp'])
    print('preprocessing and drop duplicates news:', df_copy.shape[0])
    
    df_copy = df_copy.drop(['Unnamed: 0'], axis=1)
    
    df_copy = filter_companies(df_copy)
    print('contain firm name #rows:', df_copy[df_copy['flag'] == 1].shape[0])

    return df_copy[df_copy['flag'] == 1].reset_index(drop=True)

In [ ]:
ESG_news_df = text_preprocessing_v2(news_df, types = 1)
ESG_news_df2 = text_preprocessing_v2(news_df, types = 2)

In [ ]:
# ESG_news_df = ESG_news_df[ESG_news_df['ESG'] == 1]
ESG_news_df = ESG_news_df.reset_index(drop=True)

In [ ]:
ESG_news_df = ESG_news_df2.copy()

### Fix the timestamp

In [ ]:
def timestamp_preprocessing(df, cols = 'versionCreated'):
    
    df_copy = df.copy()
    
    df_copy[cols] = df_copy[cols].apply(lambda x: x[:19])
    df_copy[cols] = pd.to_datetime(df_copy[cols])
    
    df_copy['year_month'] = df_copy[cols].apply(lambda x: datetime.strftime(x, '%Y-%m'))
    df_copy['year'] = df_copy[cols].apply(lambda x: datetime.strftime(x, '%Y'))

    return df_copy

In [ ]:
ESG_news_df = timestamp_preprocessing(ESG_news_df)

In [ ]:
ESG_news_df = ESG_news_df.sort_values(['firm','versionCreated'])
ESG_news_df.reset_index(drop=True, inplace = True)

## Convert into vector representation (Fin-Bert)

### Use ProsusAI/finbert

link: https://github.com/ProsusAI/finBERT/tree/master

In [ ]:
# Pretrained language model
model_name = 'ProsusAI/finbert'
tokenizer = BertTokenizer.from_pretrained(model_name)
language_model = BertModel.from_pretrained(model_name)

In [ ]:
print(ESG_news_df.shape[0])

In [ ]:
ESG_news_df.head(2)

In [ ]:
semantics_ftse100_df = pd.DataFrame()

for firm in ESG_news_df.firm.unique():
    
    for year in ESG_news_df[ESG_news_df.firm == firm].year.unique():
        
        copy_df = ESG_news_df[(ESG_news_df['firm'] == firm) & (ESG_news_df['year'] == year)].copy()
        esg_news = [news for news in copy_df['text_pp']]
        
        overall_semantics = []
        text_embeddings = []
        
        print(firm, year, 'started')

        for news in esg_news:
            # Sentiment extraction
            inputs = tokenizer.encode_plus(news, add_special_tokens=True, return_tensors='pt')
            outputs = language_model(**inputs)

            # Extract sentiment score from model output
            sentiment_score = outputs.pooler_output  

            # Text embedding
            text_embedding = outputs.last_hidden_state.mean(dim=1)  # Average pooling of token embeddings

            overall_semantics.append(sentiment_score)
            text_embeddings.append(text_embedding)

        # Pooling operations
        semantics_avg_vector = torch.cat(overall_semantics, dim=0).mean(dim=0)  
        text_embedding_avg_vector = torch.cat(text_embeddings, dim=0).mean(dim=0) 
        
        
        temp_semantic_df = pd.DataFrame(semantics_avg_vector.detach()).T
        temp_semantic_df['year'] = year
        temp_semantic_df['firm'] = firm
        
        semantics_ftse100_df = pd.concat([semantics_ftse100_df, temp_semantic_df])

In [ ]:
semantics_ftse100_df = pd.concat([semantics_ftse100_df[['firm','year']], \
                                  semantics_ftse100_df.drop(['firm','year'], axis=1)], axis=1)

In [ ]:
semantics_ftse100_df.head()

In [ ]:
semantics_ftse100_df = semantics_ftse100_df.reset_index(drop=True)

In [ ]:
test_df.to_csv('../data/semantics_FTSE100_esg_filter_contains.csv', index=None)

### Use yiyanghkust/finbert-pretrain

link: https://github.com/yya518/FinBERT/blob/master/finetune.ipynb

In [ ]:
# Pretrained language model
model_name = 'yiyanghkust/finbert-pretrain'
tokenizer = BertTokenizer.from_pretrained(model_name)
language_model = BertModel.from_pretrained(model_name)

In [ ]:
semantics_ftse100_v2_df = pd.DataFrame()

for firm in ESG_news_df.firm.unique():
    
    for year in ESG_news_df[ESG_news_df.firm == firm].year.unique():
        
        copy_df = ESG_news_df[(ESG_news_df['firm'] == firm) & (ESG_news_df['year'] == year)].copy()
        esg_news = [news for news in copy_df['text_pp']]
        
        overall_semantics = []
        text_embeddings = []
        
        print(firm, year, 'started')

        for news in esg_news:
            # Sentiment extraction
            inputs = tokenizer.encode_plus(news, add_special_tokens=True, return_tensors='pt')
            outputs = language_model(**inputs)

            # Extract sentiment score from model output
            sentiment_score = outputs.pooler_output  

            # Text embedding
            text_embedding = outputs.last_hidden_state.mean(dim=1)  # Average pooling of token embeddings

            overall_semantics.append(sentiment_score)
            text_embeddings.append(text_embedding)

        # Pooling operations
        semantics_avg_vector = torch.cat(overall_semantics, dim=0).mean(dim=0)  
        text_embedding_avg_vector = torch.cat(text_embeddings, dim=0).mean(dim=0) 
        
        
        temp_semantic_df = pd.DataFrame(semantics_avg_vector.detach()).T
        temp_semantic_df['year'] = year
        temp_semantic_df['firm'] = firm
        
        
        semantics_ftse100_v2_df = pd.concat([semantics_ftse100_v2_df, temp_semantic_df])

In [ ]:
semantics_ftse100_v2_df = pd.concat([semantics_ftse100_v2_df[['firm','year']], semantics_ftse100_v2_df.drop(['firm','year'], axis=1)], axis=1)

In [ ]:
semantics_ftse100_v2_df = semantics_ftse100_v2_df.reset_index(drop=True)

In [ ]:
semantics_ftse100_v2_df.head()

# ESG Data

In [ ]:
def grade_calculation_og(score):
    val = 'A+'

    if score <= 0.083333:
        val = 'D-'
    elif 0.083333 < score <= 0.166666: 
        val = 'D'
    elif 0.166666 < score <= 0.250000: 
        val = 'D+'
    elif 0.250000 < score <= 0.333333: 
        val = 'C-'
    elif 0.333333 < score <= 0.416666: 
        val = 'C'
    elif 0.416666 < score <= 0.500000: 
        val = 'C+'
    elif 0.500000 < score <= 0.583333: 
        val = 'B-'
    elif 0.583333 < score <= 0.666666: 
        val = 'B'
    elif 0.666666 < score <= 0.750000: 
        val = 'B+'
    elif 0.750000 < score <= 0.833333: 
        val = 'A-'
    elif 0.833333 < score <= 0.916666: 
        val = 'A'

    return val

In [ ]:
def grade_calculation(score):
    score = score / 100
    
    if score <= 0.250000:
        val = 'D'
    elif score <= 0.500000: 
        val = 'C'
    elif score <= 0.750000: 
        val = 'B'
    elif score > 0.75: 
        val = 'A'

    return val

In [ ]:
ESG_df = pd.read_csv('../data/output_ESGControversyScore_FTSE100.csv')

In [ ]:
ESG_df = ESG_df.iloc[:, 1:]
ESG_df['Date'] = ESG_df['Date'].astype(str)
ESG_df = ESG_df[ESG_df.Date != 'nan'].reset_index(drop=True)
ESG_df = timestamp_preprocessing(ESG_df, cols = 'Date')

In [ ]:
ESG_df.head()

In [ ]:
ESG_df['ESG Controversies Score Simple'] = ESG_df['ESG Controversies Score'].apply(lambda x: grade_calculation(x))

In [ ]:
ESG_df['ESG Controversies Score Simple'].unique()

## Match News with ESG Controversy Score

In [ ]:
semantics_ftse100_df = semantics_ftse100_ML_df.copy()

In [ ]:
for idx in range(semantics_ftse100_df.shape[0]):
    
    f = semantics_ftse100_df.iloc[idx, 0]
    y = semantics_ftse100_df.iloc[idx, 1]
    
    print(f,y)
    
    copy_df = ESG_df[(ESG_df['year'] == y) & (ESG_df['Instrument'] == f)].copy().reset_index(drop=True)
    
    if copy_df.shape[0] != 0:
    
        semantics_ftse100_df.loc[idx, 'ESG Controversies Score'] = copy_df['ESG Controversies Score'][0]
        semantics_ftse100_df.loc[idx, 'ESG Controversies Score Grade'] = copy_df['ESG Controversies Score Grade'][0]
        
    else:
        
        semantics_ftse100_df.loc[idx, 'ESG Controversies Score'] = -1
        semantics_ftse100_df.loc[idx, 'ESG Controversies Score Grade'] = 'Unknown'

## Training and Test Set

In [ ]:
train_df = semantics_ftse100_df[semantics_ftse100_df['ESG Controversies Score Grade'] != 'Unknown']
print(train_df.shape[0])

In [ ]:
test_df = semantics_ftse100_df[semantics_ftse100_df['ESG Controversies Score Grade'] == 'Unknown']
print(test_df.shape[0])

In [ ]:
train_df.to_csv('../data/ML_Data/train_footsie_contains.csv', index=None)
test_df.to_csv('../data/ML_Data/test_footsie_contains.csv', index=None)

## Data Preparation's Done

# Regression Model

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

In [ ]:
train_df = train_df.drop(['firm','year','ESG Controversies Score Grade'], axis=1).reset_index(drop=True)

In [ ]:
y = train_df.iloc[:,-1]
X = train_df.iloc[:,:-1]

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Fit PCA to Xtrn_mn
pca = PCA()

pca.fit(X)

# Calculate explained variance ratio
explained_var_ratio = pca.explained_variance_ratio_

# Calculate cumulative explained variance ratio
cumulative_exp_var_ratio = np.cumsum(explained_var_ratio)

# Plot the graph
plt.plot(range(1, len(explained_var_ratio) + 1), cumulative_exp_var_ratio)
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.show()

In [ ]:
#(3) Your code goes here

# Calculate the cumulative explained variance ratio
cumulative_explained_var_ratio = np.cumsum(pca.explained_variance_ratio_)

# Find the minimum number of principal components required to explain a certain percentage of the total variance
for percent in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    num_components = np.argmax(cumulative_explained_var_ratio >= percent) + 1
    print(f"Minimum number of principal components to explain {percent*100}% of variance: {num_components}")

In [ ]:
pca = PCA(n_components=num_components)

In [ ]:
X_pca = pca.fit_transform(X)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_pca, y, test_size=0.2, random_state=12)

In [ ]:
# Create an XGBoostRegressor model
model = XGBRegressor()

# Define the parameter grid for grid search
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of trees in the ensemble
    'learning_rate': [0.1, 0.01, 0.001],  # Learning rate
    'max_depth': [3, 4, 5]  # Maximum depth of each tree
}

In [ ]:
grid_search = GridSearchCV(model, param_grid, cv=4)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Evaluate the best model on the test set
y_pred = best_model.predict(X_val)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

# Print the best hyperparameters and evaluation metric
print("Best Hyperparameters:", grid_search.best_params_)
print("Root Mean Squared Error:", rmse)

In [ ]:
grid_search = GridSearchCV(model, param_grid, cv=4)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Evaluate the best model on the test set
y_pred = best_model.predict(X_val)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

# Print the best hyperparameters and evaluation metric
print("Best Hyperparameters:", grid_search.best_params_)
print("Root Mean Squared Error:", rmse)

In [ ]:
model = XGBRegressor(**grid_search.best_params_)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_val)
print(y_pred)

In [ ]:
y_pred_grade = [grade_calculation(sc) for sc in y_pred]

In [ ]:
y_val_grade = [grade_calculation(sc) for sc in y_val]

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy_score(y_pred_grade, y_val_grade)

In [ ]:
y_pred_grade

In [ ]:
y_val_grade

## Choose three Models (Research First)

## Cross Validation

## GridSearch

## Best Model with tuned-Hyperparams

# Numeric to Grade (Refinitiv' Rule Based)

## Predict the Test Set

# Archive

## Classification FinBert

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# create a tokenizer object
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# fetch the pretrained model 
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

def sentim_analyzer(df, tokenizer, model, headline_colum = 'headline'):
    ''' Given a df that contains a column 'headline' with article healine texts, it runs inference on the healine with the 'model' (FinBert) 
       and inserts output sentiment features into the dataframe in the respective columns (Positive_sentim, Negative_sentim, Neutral_sentim)
       
        Parameters :
          df : A dataframe that contains headlines in a column called 'headline' . 
          tokenizer(AutoTokenizer object) : A pre-processing tokenizer object from Hugging Face lib. 
          model (AutoModelForSequenceClassification object) : A hugging face transformer model.     
          
          returns df : The initial dataframe with the 3 sentiment features as columns for each headline'''
    
    for i in df.index:
        try:
            headline = df.loc[i, headline_colum]
        except:
            return print(f' \'{headline_colum}\' column might be missing from dataframe')
        # Pre-process input phrase
        
        input = tokenizer(headline, padding = True, truncation = True, return_tensors='pt')
        
        # Estimate output
        output = model(**input)
        # Pass model output logits through a softmax layer.
        predictions = torch.nn.functional.softmax(output.logits, dim=-1)
        df.loc[i, 'Positive'] = predictions[0][0].tolist()
        df.loc[i, 'Negative'] = predictions[0][1].tolist()
        df.loc[i, 'Neutral']  = predictions[0][2].tolist()
    # rearrange column order
    try:
        df = df[['date', 'stock', 'Open', 'Close', 'Volume',  'headline', 'Positive', 'Negative', 'Neutral','Price_change']]
    except:
        pass
    return df

## Filterout ESG News

In [ ]:
news_clean_df = news_df[news_df['ESG'] == 1].reset_index(drop=True)

from transformers import BertTokenizer, BertForSequenceClassification, pipeline

finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-esg',num_labels=4)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-esg')
nlp = pipeline("text-classification", model=finbert, tokenizer=tokenizer)

results = nlp(['Rhonda has been volunteering for several years for a variety of charitable community programs.',
              'Rhonda has been volunteering for several years for a variety of charitable community programs.'])
print(results) # [{'label': 'Social', 'score': 0.9906041026115417}]

news_headlines = [new for new in news_df['text_pp']]

results = nlp(news_headlines)

news_500 = pd.DataFrame(news_df['text_pp'])

for i, value in enumerate(results):
    
    news_500.loc[i:, 'label'] = value['label']
    news_500.loc[i:, 'score'] = value['score']

news_500['ESG_Maia'] = news_df['ESG']

news_500.reset_index(drop=True, inplace=True)

pd.concat([news_500[news_500['ESG_Maia'] == 1].sample(50), news_500[news_500['ESG_Maia'] == 0].sample(50)]).to_csv('/Users/yogi/Desktop/ESG_result.csv')

In [ ]:
filter out non-esg news

## FinBert-ESG

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-esg',num_labels=4)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-esg')
nlp = pipeline("text-classification", model=finbert, tokenizer=tokenizer)

results = nlp(['Rhonda has been volunteering for several years for a variety of charitable community programs.',
              'Rhonda has been volunteering for several years for a variety of charitable community programs.'])
print(results) # [{'label': 'Social', 'score': 0.9906041026115417}]

news_headlines = [new for new in news_df['text_pp']]

results = nlp(news_headlines)

news_500 = pd.DataFrame(news_df['text_pp'])

for i, value in enumerate(results):
    
    news_500.loc[i:, 'label'] = value['label']
    news_500.loc[i:, 'score'] = value['score']

news_500['ESG_Maia'] = news_df['ESG']

news_500.reset_index(drop=True, inplace=True)

pd.concat([news_500[news_500['ESG_Maia'] == 1].sample(50), news_500[news_500['ESG_Maia'] == 0].sample(50)]).to_csv('/Users/yogi/Desktop/ESG_result.csv')